# Exploring Chemical Space
On the modified ESOL dataset, some clustering techniques to explore the physicochemical diversity should be explored. Include dimensionality reduction before and visualise!

Import data

In [ ]:
import pandas as pd
import numpy as np

# Load your modified ESOL dataset
df = pd.read_csv("esol_modified.csv")

# Remove non-numeric columns (keep descriptors only)
X = df.select_dtypes(include=[np.number]).copy()

# Optional: drop target column if present (e.g., logS)
if "logS" in X.columns:
    X = X.drop(columns=["logS"])

print("Shape after numeric selection:", X.shape)

Remove low-variance features

In [ ]:
from sklearn.feature_selection import VarianceThreshold

var_filter = VarianceThreshold(threshold=0.01)
X_var = var_filter.fit_transform(X)

X = pd.DataFrame(X_var, columns=X.columns[var_filter.get_support()])
print("After variance filter:", X.shape)

Remove Highly Correlated Features (>0.9)

In [ ]:
corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X = X.drop(columns=to_drop)

print("After correlation filtering:", X.shape)

Scale the data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

PCA (for clustering space)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Original dimensions:", X_scaled.shape[1])
print("PCA dimensions:", X_pca.shape[1])

Visualisation: t-SNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

Visualisation: UMAP

In [ ]:
import umap

umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)

X_umap = umap_model.fit_transform(X_scaled)

KMeans Clustering on PCA Space

In [ ]:
# Find optimal k using silhouette score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

range_k = range(2, 9)
sil_scores = []

for k in range_k:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_pca)
    sil_scores.append(silhouette_score(X_pca, labels))

best_k = range_k[np.argmax(sil_scores)]
print("Best k:", best_k)

In [ ]:
# fit final KMeans
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
kmeans_labels = kmeans.fit_predict(X_pca)

df["KMeans_cluster"] = kmeans_labels

DBSCAN Clustering on PCA space

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.6, min_samples=5)
db_labels = dbscan.fit_predict(X_pca)

df["DBSCAN_cluster"] = db_labels

print("Number of DBSCAN clusters:",
      len(set(db_labels)) - (1 if -1 in db_labels else 0))
print("Noise points:", list(db_labels).count(-1))

Evaluation metrics

In [ ]:
from sklearn.metrics import davies_bouldin_score

print("KMeans silhouette:",
      silhouette_score(X_pca, kmeans_labels))

print("KMeans davies_bouldin_score:",
      davies_bouldin_score(X_pca, kmeans_labels))

if len(set(db_labels)) > 1:
    print("DBSCAN silhouette:",
          silhouette_score(X_pca, db_labels))

if len(set(db_labels)) > 1:
    print("DBSCAN davies_bouldin_score:",
          davies_bouldin_score(X_pca, db_labels))

Visualisation of clusters - project KMeans clustering on PCA, tSNE, UMAP

In [ ]:
import matplotlib.pyplot as plt

# PCA 2D plot
plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels)
plt.title("KMeans on PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


# t-SNE plot
plt.figure()
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels)
plt.title("KMeans projected on t-SNE")
plt.show()


# UMAP plot
plt.figure()
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=kmeans_labels)
plt.title("KMeans projected on UMAP")
plt.show()

Visualisation of clusters - project DBSCAN clustering on PCA, tSNE, UMAP

In [ ]:
# PCA 2D plot
plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels)
plt.title("DBSCAN on PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


# t-SNE plot
plt.figure()
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=db_labels)
plt.title("DBSCAN projected on t-SNE")
plt.show()


# UMAP plot
plt.figure()
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=db_labels)
plt.title("DBSCAN projected on UMAP")
plt.show()

Access Molecules from clusters via cluster labels:

In [ ]:
df["KMeans_cluster"] = kmeans_labels
df["DBSCAN_cluster"] = db_labels

In [ ]:
df

In [ ]:
# Get all molecules from KMeans cluster 0
cluster_0 = df[df["KMeans_cluster"] == 0]

print(cluster_0[["SMILES"]].head())
print("Cluster size:", len(cluster_0))

In [ ]:
# Get the DBSCAN Noise Molecules (outliers = intersting molecules?)
noise = df[df["DBSCAN_cluster"] == -1]

print("Number of outliers:", len(noise))
print(noise[["SMILES"]].head())

Identify representative molecules for KMeans clusters via centroids:

In [ ]:
from scipy.spatial.distance import cdist

cluster_id = 2

cluster_indices = np.where(kmeans_labels == cluster_id)[0]
cluster_points = X_pca[cluster_indices]

centroid = kmeans.cluster_centers_[cluster_id].reshape(1, -1)

distances = cdist(cluster_points, centroid)
closest_index = cluster_indices[np.argmin(distances)]

representative_molecule = df.iloc[closest_index]

print("Representative SMILES:", representative_molecule["SMILES"])

Visualise Molecules from certain clusters via RDKIT

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

cluster_0 = df[df["KMeans_cluster"] == 0].head(6)

mols = [Chem.MolFromSmiles(sm) for sm in cluster_0["SMILES"]]

Draw.MolsToGridImage(mols, molsPerRow=3)

Compare clusters accross methods (where is the noise located? are clusters consistent?)

In [ ]:
pd.crosstab(df["KMeans_cluster"], df["DBSCAN_cluster"])

Cluster Exploration: Compare for each cluster

1) Report cluster size
2) Report centroid representative molecule
3) Show 4–6 example structures
4) Report key descriptor means
5) Compare to global dataset mean

In [ ]:
summary = df.groupby("KMeans_cluster").agg({
    "LogS": "mean",
    "MolWt": "mean",
    "LogP": "mean",
    "NumHDonors": "mean",
    "NumHAcc": "mean"
})

print(summary)

In [ ]:
summary2 = df.groupby("KMeans_cluster")["LogS"].mean()
print(summary2)